# Assignment 3 · Track C：结果查看与复现

作者：Jiabao Gao · 学号：122090121

项目包含 R0–R4、两组消融和最终测试。最终方案的 dev 准确率为 56%，test 为 43%。
本 notebook 默认读取 `runs/main` 中保存的结果。第 5 节提供 dev 复现入口。


## 1. 定位项目

本地在项目目录启动 Jupyter；Colab 先上传并解压完整 ZIP。将 PROJECT 设为含有 run_trackc.py 的目录。


In [ ]:
from pathlib import Path
import json, csv, sys, subprocess
PROJECT = Path('.').resolve()
if not (PROJECT / 'run_trackc.py').exists() and (PROJECT / 'NLP_Assignment3_TrackC').is_dir():
    PROJECT = PROJECT / 'NLP_Assignment3_TrackC'
assert (PROJECT / 'run_trackc.py').exists(), '请将 PROJECT 指向已解压的项目目录'
RUN = PROJECT / 'runs' / 'main'
def load(name):
    return json.loads((RUN / name).read_text(encoding='utf-8'))
print(PROJECT)


## 2. 查看实测指标与 RSI 曲线


In [ ]:
with (RUN / 'complete_results.csv').open(encoding='utf-8-sig', newline='') as f:
    for row in csv.DictReader(f):
        print(f"{row['split']:4} {row['artifact']:24} accuracy={float(row['accuracy']):.0%} "
              f"tokens={row['generated_tokens']} seconds={float(row['elapsed_seconds']):.1f}")
try:
    from IPython.display import display, Image
    display(Image(filename=str(RUN / 'rsi_curve.png')))
except ImportError:
    print('Curve:', RUN / 'rsi_curve.png')


## 3. 查看案例

报告中的案例包括：R1 的算术题 5878、R2 的余额题 2437、R3 的小费题 4632，以及 R4 的罐头题 819 和豆子题 3115。
修改 ROUND 和 CASE_ID 可以查看相应输出。ID 对应原始训练集索引。


In [ ]:
ROUND = 'R4'
CASE_ID = 819
row = next(r for r in load(ROUND + '.json')['records'] if r['id'] == CASE_ID)
print(json.dumps(row, ensure_ascii=False, indent=2))


## 4. 查看最终测试与配置

以下单元格显示最终选定的配置、dev/test 分数及冻结时间。


In [ ]:
assert (RUN / 'test_started.json').exists()
final = load('final_test.json')
print('Selected:', final['selected_round'])
print('Dev:', final['dev_accuracy'], 'Test:', final['metrics']['accuracy'])
print('Gap (percentage points):', round(final['gap_percentage_points'], 1))
print('Frozen at:', load('frozen.json')['frozen_utc'])


## 5. 复现 dev 配置

按 README 安装 PyTorch 和 requirements.txt，并使用 CUDA GPU。将 REPRODUCE_DEV 设为 True 后运行；结果会写入新的 reproduction 目录。
复现使用保存的提示词，无需 improver API。硬件和 CUDA kernel 差异可能影响采样结果。


In [ ]:
REPRODUCE_DEV = False
if REPRODUCE_DEV:
    subprocess.run([sys.executable, str(PROJECT / 'reproduce_dev.py'),
                    '--rounds', 'R0', 'R1', 'R2', 'R3', 'R4',
                    'ablation_fixed3', 'ablation_fixed5'], cwd=PROJECT, check=True)
else:
    print('Set REPRODUCE_DEV to True to run the saved configurations.')


## 6. 开展新的 API 搜索实验

按 README 的阶段流程创建新实验，统一使用 `--run-dir runs/new_experiment`。各轮计划依据新实验中的错误记录填写。

本次使用 Base URL `https://api.deepseek.com` 和模型 `deepseek-flash`，关闭 thinking 并请求 JSON 输出。
密钥通过 `IMPROVER_API_KEY` 环境变量读取；Windows 也支持同名用户变量，Colab 可使用 Secrets。
以下为配置示例。


In [ ]:
# import os
# os.environ['IMPROVER_BASE_URL'] = 'https://api.deepseek.com'
# os.environ['IMPROVER_MODEL'] = 'deepseek-flash'
# Colab only:
# from google.colab import userdata
# os.environ['IMPROVER_API_KEY'] = userdata.get('IMPROVER_API_KEY')


## 7. 提交报告与代码

报告已署名 Jiabao Gao（122090121）。登录学校 Blackboard，进入本课程 Assignment 3 页面提交报告，并按课程要求附上 GitHub 源码链接和代码附件。具体步骤见 README。

作业 PDF 的截止时间为 2026-12-11 23:59，课程官网为 2026-12-04 23:59；请核对 Blackboard 本次作业页面。
